In [ ]:
# ============ IMPORTS ============
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import quantstats as qs
from src.calculate_ml_metrics import calculate_ml_metrics
from src.calculate_trade_metrics import calculate_trade_metrics
from xgboost import XGBClassifier

# =========== CONFIGURATION ============
feature_set = "confirmed" # "all" or "confirmed" -  features confirmed by boruta algorithm
window = 7
model_names = ['log_reg', 'random_forest', 'svc','xgboost']
# ======================================

In [ ]:
# =========== MLFLOW SETUP ============
experiment  = mlflow.get_experiment_by_name("triple_barrier_classification")
experiment_id = experiment.experiment_id

# =========== LIST PREVIOUS RUNS ============
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs = runs.sort_values("start_time", ascending=False)
runs = runs[runs["status"] == "FINISHED"]  # Filter for finished runs

# ========== LOAD LATEST MODELS ==========
latest_run_df = runs.groupby('tags.mlflow.runName').first()

In [ ]:
# =========== LOAD MODELS AND PREDICTIONS ============
models = {}
train_predictions = {}
test_predictions = {}

for model_name in model_names:
    run_name = f"{model_name}_{feature_set}_features"
    try:
        run_id = latest_run_df.loc[run_name, "run_id"]

        models[model_name] = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

        train_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"final_model_train_data_{model_name}_{feature_set}.pkl"
        )
        test_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"final_model_test_data_{model_name}_{feature_set}.pkl"
        )

        train_predictions[model_name] = pd.read_pickle(train_path)
        test_predictions[model_name]  = pd.read_pickle(test_path)

    except KeyError:
        print(f"No finished run found for '{run_name}' — skipping.")

In [ ]:
# =========== EVALUATION ============
ml_metrics = []

for model_name in model_names:
    try:
        train_df = train_predictions[model_name]
        test_df = test_predictions[model_name]
        y_train = train_df[f"Label_{window}day"]
        train_pred = train_df[f"pred_{model_name}"]
        train_proba = train_df[f"proba_{model_name}"]
        y_test = test_df[f"Label_{window}day"]
        test_pred = test_df[f"pred_{model_name}"]
        test_proba = test_df[f"proba_{model_name}"]
        
        train_ml_metrics_dict = calculate_ml_metrics(y_train, train_pred, train_proba, "train")
        test_ml_metrics_dict = calculate_ml_metrics(y_test, test_pred, test_proba, "test")
        ml_metrics.append({"Model": model_name, **train_ml_metrics_dict, **test_ml_metrics_dict})
    except KeyError:
        print(f"No stored dataframes for '{model_name}' — skipping.")
        
ml_metrics = pd.DataFrame(ml_metrics).set_index("Model").T

# Log with run

In [ ]:
# ============ TRADING METRICS ============
trade_metrics = []

for model_name in model_names:
    try:
        train_df = train_predictions[model_name]
        test_df  = test_predictions[model_name]

        y_train  = train_df[f"Label_{window}day"]
        y_test   = test_df[f"Label_{window}day"]
        train_pred = train_df[f"pred_{model_name}"]
        test_pred  = test_df[f"pred_{model_name}"]

        actual_returns_train = train_df[f"Return_{window}day"]
        actual_returns_test  = test_df[f"Return_{window}day"]

        train_trade_metrics_dict = calculate_trade_metrics(y_train, train_pred, actual_returns_train, window, "train")
        test_trade_metrics_dict  = calculate_trade_metrics(y_test,  test_pred,  actual_returns_test,  window, "test")

        trade_metrics.append({"Model": model_name, **train_trade_metrics_dict, **test_trade_metrics_dict})
    except KeyError:
        print(f"No stored dataframes for '{model_name}' — skipping.")

trade_metrics = pd.DataFrame(trade_metrics).set_index("Model").T

        
# # ============ SHOW & STORE RESULTS ============
# trade_metrics = pd.DataFrame(trade_metrics)
# print(trade_metrics.to_markdown(index=False))
# trade_metrics.to_csv('data_cache/trade_metrics.csv', index=False)

# HTML Report with QuantStats
# returns_series = pd.Series(returns_decimal.values, index=X_test.index[traded_mask])
# qs.reports.html(returns_series, output='data_cache/model_performance_tearsheet.html', 
#                 title=f'{model_name} Trading Performance')

In [ ]:
# # =========== HELPER FUNCTION ===========
# def get_evaluator_config(model_name):
#     base = {"log_explainer": True,                      # Save explainer as MLflow artifact.
#             "explainer_type": "auto",           
#             # --- SHAP computation controls ---
#             "explainability_nsamples": 200,             # Rows used for SHAP value estimation
#             "explainability_kernel_link": "identity",   # identity, logit (log-odds)
#             # --- Output controls ---
#             "log_model_explanations": False,            # Log per-row SHAP values, computationally expensive
#             "max_error_examples": 50,                   # How many misclassified examples to explain
#     }
#     if model_name == "log_reg":                         # Configure shap per model if needed.
#         base["explainer_type"] = "linear"
#     elif model_name == "svc":
#         base["explainability_nsamples"] = 15
#     return base

# # ============ MLFLOW EVALUATE (for plots only) ============    
#         # Create evaluation dataset (X_test + targets)
#         eval_data = X_test.copy()
#         if model_name == "xgboost":
#             eval_data[f"Label_{window}day"] = y_test.map(label_map)
#         else:
#             eval_data[f"Label_{window}day"] = y_test
        
#         # Evaluate with MLflow - generates all metrics & plots automatically
#         eval_result = mlflow.models.evaluate(
#             model=model_info.model_uri,
#             data=eval_data,
#             targets=f"Label_{window}day",
#             model_type="classifier",
#             evaluator_config=get_evaluator_config(model_name)
#         )
        
# Log ml metrics and log trade metrics